In [132]:
import pandas as pd
import numpy as np
from pathlib import Path

## Part zero configuration

In [133]:
ROOT_DIR=Path.cwd().parent.parent.parent
DATASET_DIR= ROOT_DIR / "dataset"
CSV_FILE_PATH = DATASET_DIR / "raw_loan_dataset.csv"
print(ROOT_DIR)
print(DATASET_DIR)
print(CSV_FILE_PATH)

/home/mdev/Documents/ml/ds-ml-bootcamp
/home/mdev/Documents/ml/ds-ml-bootcamp/dataset
/home/mdev/Documents/ml/ds-ml-bootcamp/dataset/raw_loan_dataset.csv


## Load & Inspect

In [134]:
df = pd.read_csv(CSV_FILE_PATH)
df = df.rename(columns={'Approved':'Status'})
df.head()

,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Status
0,108810,537.0,1.1,25800,Yes,No,No
1,96482,524.0,15.0,11200,Y,No,Yes
2,28478,NaN,5.4,12100,No,No,Yes
3,"$25,851",561.0,17.6,7000,Yes,No,Yes
4,38396,527.0,9.8,10700,No,No,Approved


In [135]:
rows,columns = df.shape
print("ROWS:",rows)
print("COLUMNS:",columns)

ROWS: 103
COLUMNS: 7


In [136]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     object 
 1   CreditScore       97 non-null     float64
 2   EmploymentYears   98 non-null     float64
 3   LoanAmount        98 non-null     object 
 4   HasCollateral     101 non-null    object 
 5   PreviousDefaults  101 non-null    object 
 6   Status            103 non-null    object 
dtypes: float64(2), object(5)
memory usage: 5.8+ KB


In [137]:
df.isna().sum()

Income              5
CreditScore         6
EmploymentYears     5
LoanAmount          5
HasCollateral       2
PreviousDefaults    2
Status              0
dtype: int64

In [138]:
def display_unique_values(col:str):
    print(f"\n{col}:",df[col].unique())

In [139]:
categorical_cols = ['Status','HasCollateral','PreviousDefaults']
for col in categorical_cols:
    display_unique_values(col) # use to capture and understand typo errors and in consistence category


Status: ['No' 'Yes' 'Approved' 'Rejected' 'approved' 'rejected' 'YES' 'NO']

HasCollateral: ['Yes' 'Y' 'No' 'N' nan 'yse' 'yes']

PreviousDefaults: ['No' nan 'Yes' '1' '0' 'Y' 'N']


## Clean Currency Formatting

In [140]:
def clean_numeric_col(col:str):
    print(f"\nCleaning column: {col}")
    df[col] = df[col].replace(r'[^0-9.]','',regex=True).astype(str)
    df[col] = df[col].str.strip()
    df[col] = df[col].replace('',np.nan)
    df[col] = pd.to_numeric(df[col],errors="coerce")

In [141]:
currency_cols = ['Income','LoanAmount']
for col in currency_cols:
    clean_numeric_col(col)
    print(df[col].head())


Cleaning column: Income
0    108810.0
1     96482.0
2     28478.0
3     25851.0
4     38396.0
Name: Income, dtype: float64

Cleaning column: LoanAmount
0    25800.0
1    11200.0
2    12100.0
3     7000.0
4    10700.0
Name: LoanAmount, dtype: float64


## Fix Category Errors before Imputation

In [142]:
categorical_map = {
    '1':'Yes','Approved':'Yes','approved':'Yes','Y':'Yes','yse':'Yes',
    'Rejected':'No','rejected':'No','N':'No','0':'No'
    
}

def clean_format_categorical_col(col:str,categorical_map:dict):
    print(f"\nCleaning column: {col}")
    df[col] = df[col].replace(categorical_map).astype(str)
    df[col] = df[col].str.strip().str.title()
    df[col] = df[col].replace({'Nan':np.nan})

In [143]:
for col in categorical_cols:
    display_unique_values(col)
    clean_format_categorical_col(col,categorical_map)
    display_unique_values(col)


Status: ['No' 'Yes' 'Approved' 'Rejected' 'approved' 'rejected' 'YES' 'NO']

Cleaning column: Status

Status: ['No' 'Yes']

HasCollateral: ['Yes' 'Y' 'No' 'N' nan 'yse' 'yes']

Cleaning column: HasCollateral

HasCollateral: ['Yes' 'No' nan]

PreviousDefaults: ['No' nan 'Yes' '1' '0' 'Y' 'N']

Cleaning column: PreviousDefaults

PreviousDefaults: ['No' nan 'Yes']


## Impute Missing Values

In [144]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     float64
 1   CreditScore       97 non-null     float64
 2   EmploymentYears   98 non-null     float64
 3   LoanAmount        98 non-null     float64
 4   HasCollateral     101 non-null    object 
 5   PreviousDefaults  101 non-null    object 
 6   Status            103 non-null    object 
dtypes: float64(4), object(3)
memory usage: 5.8+ KB


In [145]:
df.isna().sum()

Income              5
CreditScore         6
EmploymentYears     5
LoanAmount          5
HasCollateral       2
PreviousDefaults    2
Status              0
dtype: int64

In [146]:
def fill_with_median(col:str):
    print(f"\nFilling column: {col}")
    df[col] = df[col].fillna(df[col].median())
        
def fill_with_mode(col:str):
    print(f"\nFilling column: {col}")
    df[col] = df[col].fillna(df[col].mode()[0])

In [147]:
numeric_cols = currency_cols + ['CreditScore','EmploymentYears']
print(currency_cols)
print(numeric_cols)

['Income', 'LoanAmount']
['Income', 'LoanAmount', 'CreditScore', 'EmploymentYears']


In [148]:
for col in numeric_cols:
    fill_with_median(col)


Filling column: Income

Filling column: LoanAmount

Filling column: CreditScore

Filling column: EmploymentYears


In [149]:
for col in categorical_cols:
    fill_with_mode(col)
    display_unique_values(col)


Filling column: Status

Status: ['No' 'Yes']

Filling column: HasCollateral

HasCollateral: ['Yes' 'No']

Filling column: PreviousDefaults

PreviousDefaults: ['No' 'Yes']


In [150]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            103 non-null    float64
 1   CreditScore       103 non-null    float64
 2   EmploymentYears   103 non-null    float64
 3   LoanAmount        103 non-null    float64
 4   HasCollateral     103 non-null    object 
 5   PreviousDefaults  103 non-null    object 
 6   Status            103 non-null    object 
dtypes: float64(4), object(3)
memory usage: 5.8+ KB


In [151]:
df.isna().sum()

Income              0
CreditScore         0
EmploymentYears     0
LoanAmount          0
HasCollateral       0
PreviousDefaults    0
Status              0
dtype: int64

## Remove Duplicates

In [152]:
before = df.shape
df = df.drop_duplicates()
print(f"Before dropping {before}(rows,cols) after dropped {df.shape}(rows,cols)")

Before dropping (103, 7)(rows,cols) after dropped (100, 7)(rows,cols)
